# 현실의 데이터 22: 반도체 공정 불량 예측

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/datashop/22_semiconductor_yield.ipynb)

**분야**: 제조 &nbsp;|&nbsp; **이 분야에서 먼저 보는 문제**: 수율 · 불량 예측 · 센서 데이터 · 공정 이상 탐지

> 반도체 생산 라인 센서 590개, 제품 1,567개. 센서만 보고 불량을 미리 알 수 있을까?

**데이터**: SECOM (약 2MB). UCI (McCann & Johnston, 2008), 반도체 제조 공정 센서 590개 × 1,567개 제품, 불량 104개. 센서 이름은 공개되지 않음.

인턴 · 산학협력 과제에서 처음 받게 될 데이터와 닮은 공개 데이터다. 데이터를 열어 보고, 아래 예시처럼 문제 하나를 정해 기준선과 비교한 뒤, 맨 끝의 **나의 다섯 칸** 을 채운다 (1주 2차시 '문제 정형화와 평가지표').

| 칸 | 이 노트북의 예시 |
|---|---|
| ① 한 행 | 제품 (웨이퍼) 하나 |
| ② 입력 X | 센서 값 590개 (빈칸 많음) |
| ③ 출력 y | 최종 검사 불량인가 |
| ④ 평가지표 | 재현율 · 정밀도 (불량을 놓치지 않기) |
| ⑤ 기준선 | 전부 '양품' |

**논문 · 출처** (직접 열어 보자: 이 데이터를 누가, 왜, 어떻게 만들었나):
- [UCI SECOM 데이터 설명 (McCann & Johnston, 2008)](https://archive.ics.uci.edu/dataset/179/secom)

In [1]:
import io, os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url, path):
    """Download url to path once (skip if it already exists)"""
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if not os.path.exists(path):
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (ML course data shop)"})
        with urllib.request.urlopen(req, timeout=600) as r, open(path, "wb") as f:
            f.write(r.read())
    return path

## 입장: 데이터 받기

In [2]:
URL = "https://archive.ics.uci.edu/static/public/179/secom.zip"
with zipfile.ZipFile(download(URL, "data/secom.zip")) as z:
    X = pd.read_csv(z.open("secom.data"), sep=" ", header=None)
    labels = pd.read_csv(z.open("secom_labels.data"), sep=" ", header=None, names=["label", "time"])
X.columns = [f"s{i}" for i in range(X.shape[1])]
y = (labels["label"] == 1).astype(int).values
time = pd.to_datetime(labels["time"], format="%d/%m/%Y %H:%M:%S")
print(X.shape, f"fail rate {y.mean():.1%}", time.min(), "~", time.max())

(1567, 590) fail rate 6.6% 2008-07-19 11:55:00 ~ 2008-10-17 06:07:00


## 열어 보기: 공장 데이터는 지저분하다

빈칸은 얼마나? 값이 하나뿐인 (쓸모없는) 센서는? (1주 3차시 '데이터 직접 다루기' 와 같은 점검)

In [3]:
missing = X.isna().mean()
print(f"cells missing: {X.isna().mean().mean():.1%}")
print("sensors more than half missing:", (missing > 0.5).sum())
print("sensors with a single value:", (X.nunique() <= 1).sum())
print(X.describe().T[["min", "max", "mean"]].sample(8, random_state=0).round(2))

cells missing: 4.5%
sensors more than half missing: 28
sensors with a single value: 116
         min      max     mean
s225  168.80  3601.30  1041.06
s14     2.25    19.55     9.01
s85     0.11     0.12     0.11
s418    0.00   999.32   320.26
s132    2.19     2.47     2.32
s213    0.00     1.28     0.06
s186    0.00     0.00     0.00
s340    0.01  5043.88    14.67


## 문제 하나 골라 보기: 제품 하나 → 불량일까

**시간 순서** 로 앞 70% 학습, 뒤 30% 시험. 청소: 반 이상 빈 센서 · 값이 하나뿐인 센서는 버리고, 남은 빈칸은 학습 데이터의 중앙값으로. 불량이 드물어서 모델에 '불량 쪽을 더 중요하게' (`class_weight="balanced"`) 를 준다.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def precision_recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1)); fp = np.sum((y_pred == 1) & (y_true == 0)); fn = np.sum((y_pred == 0) & (y_true == 1))
    return (tp / (tp + fp) if tp + fp else 0.0), tp / (tp + fn)

order = np.argsort(time.values); cut = int(len(order) * 0.7)
tr, te = order[:cut], order[cut:]
keep = [c for c in X.columns if X[c].iloc[tr].isna().mean() <= 0.5 and X[c].iloc[tr].nunique() > 1]
med = X[keep].iloc[tr].median()
A, B = X[keep].iloc[tr].fillna(med), X[keep].iloc[te].fillna(med)
print("sensors kept:", len(keep), "| test fails:", y[te].sum(), "of", len(te))

from sklearn.metrics import roc_auc_score
m = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, class_weight="balanced", max_iter=5000)).fit(A, y[tr])
for name, pred in [("all good", np.zeros(len(te), dtype=int)), ("logistic (balanced)", m.predict(B))]:
    p, r = precision_recall(y[te], pred)
    print(f"{name:20s} accuracy {np.mean(pred == y[te]):.1%}  precision {p:.0%}  recall {r:.0%}")

# A factory can re-inspect only some wafers: flag the 20% the model finds most suspicious
score = m.predict_proba(B)[:, 1]
flag = (score >= np.quantile(score, 0.8)).astype(int)
print(f"\nre-inspect the top 20%: caught {np.sum(flag & y[te])} of {y[te].sum()} fails "
      f"(re-inspecting a random 20% would catch about {0.2 * y[te].sum():.0f})")
print(f"ROC-AUC {roc_auc_score(y[te], score):.2f}  (0.5 = coin flip)")

sensors kept: 444 | test fails: 26 of 471
all good             accuracy 94.5%  precision 0%  recall 0%
logistic (balanced)  accuracy 90.7%  precision 0%  recall 0%

re-inspect the top 20%: caught 6 of 26 fails (re-inspecting a random 20% would catch about 5)
ROC-AUC 0.59  (0.5 = coin flip)


**관찰**: 센서가 590개나 있어도 이 모델은 불량을 거의 못 맞힌다 (무작위로 고른 것과 비슷하다). 현실 데이터에서 흔한 결말이다. 기준선과 비교했기 때문에 '모델이 쓸모 있다' 고 착각하지 않을 수 있다.
다음에 할 일은 모델을 키우는 것보다 **문제를 다시 정하는 것** 일 때가 많다: 불량을 더 잘 설명하는 공정 단계의 데이터를 받거나, 엔지니어와 함께 의심 센서를 고르거나, '불량' 의 정의를 나누는 것.
그리고 공장에서는 '놓친 불량 하나' 와 '괜히 다시 검사한 양품 하나' 의 비용을 비교해 재검사 비율을 정한다.

## 다른 문제로 바꿔 보기

- **원인 찾기**: '불량일까' 대신 '불량과 가장 관련 있는 센서 10개' 를 출력으로 정하면? 지표는 무엇일까? (엔지니어가 확인해 줘야 한다)
- **이상 탐지**: 불량 라벨이 없다면? 평소와 다른 제품을 찾는 문제로 바꿀 수 있다 (4주차 k-means · 가우시안).

### 나의 다섯 칸

아래 표를 직접 채워 보자. 위의 예시와 다르게 정해도 된다.

| 칸 | 내가 정한 것 |
|---|---|
| ① 한 행은 무엇? | |
| ② 입력 X (예측할 때 알 수 있는 것만) | |
| ③ 출력 y | |
| ④ 평가지표 | |
| ⑤ 기준선 | |